In [1]:
import json
import os
import requests
from openai import OpenAI
from pydantic import BaseModel, Field

In [2]:
try:
    client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))
except Exception as e:
    print(f"Error initializing OpenAI client: {e}")
    print("Please make sure your OPENAI_API_KEY environment variable is set.")
    exit()

In [6]:
# Define the functions

In [3]:
def list_files(directory="."):
    """Lists all files and subdirectories in a given directory."""
    print(f"[Agent Action: Listing files in '{directory}']")
    try:
        files = os.listdir(directory)
        return {"files": files}
    except Exception as e:
        return {"error": str(e)}

In [4]:
def read_file(filename):
    """Reads the entire content of a specified file."""
    print(f"[Agent Action: Reading file '{filename}']")
    try:
        with open(filename, "r") as f:
            content = f.read()
        return {"content": content}
    except Exception as e:
        return {"error": str(e)}

In [5]:
def write_file(filename, content):
    """Writes or overwrites content to a specified file."""
    print(f"[Agent Action: Writing to file '{filename}']")
    try:
        with open(filename, "w") as f:
            f.write(content)
        return {"status": "success", "filename": filename}
    except Exception as e:
        return {"error": str(e)}

In [7]:
# Describing the tools

In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files and directories in a given directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {
                        "type": "string",
                        "description": "The directory to list. Defaults to '.' (current directory).",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a specified file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "The name (including path) of the file to read.",
                    },
                },
                "required": ["filename"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write or overwrite content to a specified file. Creates the file if it doesn't exist.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "The name (including path) of the file to write to.",
                    },
                    "content": {
                        "type": "string",
                        "description": "The text content to write into the file.",
                    },
                },
                "required": ["filename", "content"],
            },
        },
    },
]

In [10]:
def call_function(name, args):
    if name == "list_files":
        return list_files(**args)
    elif name == "read_file":
        return read_file(**args)
    elif name == "write_file":
        return write_file(**args)
    else:
        return {"error": f"Unknown function: {name}"}

In [11]:
messages = [
    {"role": "system", "content": "You are a helpful coding assistant. You have tools to list, read, and write files on the local filesystem. Always confirm the action taken."},
    {"role": "user", "content": "Please create a file called 'hello.txt' and write 'Hello from my first coding agent!' inside it."},
]

In [12]:
print(f"User: {messages[1]['content']}\n")

User: Please create a file called 'hello.txt' and write 'Hello from my first coding agent!' inside it.



In [13]:
#Call the model

In [14]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [ ]:
#The code above doesn't actually execute what we want. It just decides which
#function to call

In [15]:
completion.model_dump()

{'id': 'chatcmpl-CUvpOtypI57jKQqpntKqfQZPUDusN',
 'choices': [{'finish_reason': 'tool_calls',
   'index': 0,
   'logprobs': None,
   'message': {'content': None,
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_lK0HmR7xFtdKZTBMBkYYyzFn',
      'function': {'arguments': '{"filename":"hello.txt","content":"Hello from my first coding agent!"}',
       'name': 'write_file'},
      'type': 'function'}]}}],
 'created': 1761488166,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 226,
  'prompt_tokens': 288,
  'total_tokens': 514,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 192,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}

In [16]:
choice = completion.choices[0]
print(choice)

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_lK0HmR7xFtdKZTBMBkYYyzFn', function=Function(arguments='{"filename":"hello.txt","content":"Hello from my first coding agent!"}', name='write_file'), type='function')]))


In [17]:
if not choice.message.tool_calls:
    print("Agent: The model didn't request a tool call. Here's its response:")
    print(choice.message.content)
else:
    # Add the agent's request to call a tool to the message history
    messages.append(choice.message)

    # Execute all tool calls requested
    for tool_call in choice.message.tool_calls:
        print(f"Agent: Needs to call tool '{tool_call.function.name}' with args {tool_call.function.arguments}")
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        # Call the actual Python function
        result = call_function(name, args)
        
        # Add the tool's output to the message history
        messages.append(
            {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
        )

Agent: Needs to call tool 'write_file' with args {"filename":"hello.txt","content":"Hello from my first coding agent!"}
[Agent Action: Writing to file 'hello.txt']


In [18]:
messages

[{'role': 'system',
  'content': 'You are a helpful coding assistant. You have tools to list, read, and write files on the local filesystem. Always confirm the action taken.'},
 {'role': 'user',
  'content': "Please create a file called 'hello.txt' and write 'Hello from my first coding agent!' inside it."},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_lK0HmR7xFtdKZTBMBkYYyzFn', function=Function(arguments='{"filename":"hello.txt","content":"Hello from my first coding agent!"}', name='write_file'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_lK0HmR7xFtdKZTBMBkYYyzFn',
  'content': '{"status": "success", "filename": "hello.txt"}'}]

In [19]:
completion_2 = client.chat.completions.create(
        model="gpt-4-turbo",
        messages=messages,
        tools=tools,
    )

In [20]:
print(f"\nAgent: {completion_2.choices[0].message.content}")


Agent: The file `hello.txt` has been created and the message "Hello from my first coding agent!" has been written inside it.
